In [164]:
from datasets import load_dataset
ds = load_dataset("bentrevett/multi30k")

In [165]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import copy

In [166]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [167]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, device):
        super().__init__()
        encoding = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        _2i = torch.arange(0, d_model, 2).float()
        encoding[:, 0::2] = torch.sin(pos / (10000 ** (_2i / d_model)))
        encoding[:, 1::2] = torch.cos(pos / (10000 ** (_2i / d_model)))
        self.register_buffer('encoding', encoding)

    def forward(self, x):
        batch_size, seq_len = x.size()
        return self.encoding[:seq_len, :]

In [168]:
class TokenEmbedding(nn.Embedding):
    def __init__(self,vocab_size,d_model,padding_idx=1):
        super().__init__(vocab_size,d_model,padding_idx=padding_idx)

In [169]:
class ScaleDotProductAttention(nn.Module):
    def __init__(self):
        super(ScaleDotProductAttention,self).__init__()
        self.softmax=nn.Softmax(dim=-1)
    def forward(self,q,k,v,mask=None):
        batch_size,head,length,d_tensor=k.size()
        k_t=k.transpose(2,3)
        score=(q@k_t)/math.sqrt(d_tensor)
        if mask is not None:
            score=score.masked_fill(mask==0,-10000)
        score=self.softmax(score)
        v=score@v
        return v,score
        

In [170]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,n_head):
        super(MultiHeadAttention,self).__init__()
        self.n_head=n_head
        self.attention=ScaleDotProductAttention()
        self.w_q=nn.Linear(d_model,d_model)
        self.w_k=nn.Linear(d_model,d_model)
        self.w_v=nn.Linear(d_model,d_model)
        self.w_concat=nn.Linear(d_model,d_model)
    def split(self,tensor):
        batch_size,length,d_model=tensor.size()
        d_tensor=d_model//self.n_head
        tensor=tensor.view(
            batch_size,
            length,
            self.n_head,
            d_tensor
        ).transpose(1,2)
        return tensor
    def concat(self,tensor):
        batch_size,head,length,d_tensor=tensor.size()
        d_model=head*d_tensor
        tensor=tensor.transpose(1,2).contiguous().view(
            batch_size,
            length,
            d_model
        )
        return tensor
    
    def forward(self,q,k,v,mask=None):
        q,k,v=self.w_q(q),self.w_k(k),self.w_v(v)
        q,k,v=self.split(q),self.split(k),self.split(v)
        out,attention=self.attention(q,k,v,mask=mask)
        out=self.concat(out)
        out=self.w_concat(out)
        return out

In [171]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self,d_model,hidden,dropout=0.1):
        super(PositionwiseFeedForward,self).__init__()
        self.linear1=nn.Linear(d_model,hidden)
        self.linear2=nn.Linear(hidden,d_model)
        self.relu=nn.ReLU()
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        x=self.linear1(x)
        x=self.relu(x)
        x=self.dropout(x)
        x=self.linear2(x)
        return x

In [172]:
class LayerNorm(nn.Module):
    def __init__(self,d_model,eps=1e-12):
        super(LayerNorm,self).__init__()
        self.gamma=nn.Parameter(torch.ones(d_model))
        self.beta=nn.Parameter(torch.zeros(d_model))
        self.eps=eps
    def forward(self,x):
        mean=x.mean(-1,keepdim=True)
        var=x.var(-1,unbiased=False,keepdim=True)
        out=(x-mean)/torch.sqrt(var+self.eps)
        out=self.gamma*out+self.beta
        return out

In [173]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, hidden,n_head,dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.attention = MultiHeadAttention(d_model=d_model, n_head=n_head)
        self.norm1 = LayerNorm(d_model=d_model)
        self.dropout1 = nn.Dropout(dropout)
        
        self.ffn = PositionwiseFeedForward(d_model=d_model,hidden=hidden,dropout=dropout)
        self.norm2 = LayerNorm(d_model=d_model)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        _x = x
        x = self.attention(q=x, k=x, v=x, mask=src_mask)
        x = self.dropout1(x)
        x = self.norm1(x + _x)
        _x = x
        x = self.ffn(x)
        x = self.dropout2(x)
        x = self.norm2(x + _x)
        return x
        

In [174]:
class DecoderLayer(nn.Module):

    def __init__(self, d_model, hidden, n_head, dropout=0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_head)
        self.norm1 = LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.enc_dec_attention = MultiHeadAttention(d_model, n_head)
        self.norm2 = LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
        self.ffn = PositionwiseFeedForward(
            d_model=d_model,
            hidden=hidden,
            dropout=dropout
        )

        self.norm3 = LayerNorm(d_model)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, dec, enc, trg_mask, src_mask):

        _x = dec
        x = self.self_attention(
            q=dec,
            k=dec,
            v=dec,
            mask=trg_mask
        )

        x = self.dropout1(x)

        x = self.norm1(x + _x)

        _x = x

        x = self.enc_dec_attention(
            q=x,
            k=enc,
            v=enc,
            mask=src_mask
        )

        x = self.dropout2(x)

        x = self.norm2(x + _x)

        _x = x

        x = self.ffn(x)

        x = self.dropout3(x)

        x = self.norm3(x + _x)

        return x

In [175]:
class Encoder(nn.Module):

    def __init__(
        self,
        enc_voc_size,
        max_len,
        d_model,
        hidden,
        n_head,
        n_layers,
        dropout,
        device
    ):
        super().__init__()

        self.embedding = TokenEmbedding(enc_voc_size, d_model)

        self.pos_encoding = PositionalEncoding(
            d_model,
            max_len,
            device
        )

        self.layers = nn.ModuleList([
            EncoderLayer(
                d_model=d_model,
                hidden=hidden,
                n_head=n_head,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):

        emb = self.embedding(x)

        pos = self.pos_encoding(x)

        x = self.dropout(emb + pos)

        for layer in self.layers:
            x = layer(x, src_mask)

        return x

In [176]:
class Decoder(nn.Module):

    def __init__(
        self,
        dec_voc_size,
        max_len,
        d_model,
        hidden,
        n_head,
        n_layers,
        dropout,
        device
    ):
        super().__init__()

        self.embedding = TokenEmbedding(dec_voc_size, d_model)

        self.pos_encoding = PositionalEncoding(
            d_model,
            max_len,
            device
        )

        self.layers = nn.ModuleList([
            DecoderLayer(
                d_model=d_model,
                hidden=hidden,
                n_head=n_head,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        self.fc = nn.Linear(d_model, dec_voc_size)

        self.dropout = nn.Dropout(dropout)

    def forward(self, dec, enc, trg_mask, src_mask):

        emb = self.embedding(dec)

        pos = self.pos_encoding(dec)

        x = self.dropout(emb + pos)

        for layer in self.layers:
            x = layer(x, enc, trg_mask, src_mask)

        output = self.fc(x)

        return output

In [177]:
class Transformer(nn.Module):

    def __init__(
        self,
        src_pad_idx,
        trg_pad_idx,
        trg_sos_idx,
        enc_voc_size,
        dec_voc_size,
        d_model,
        hidden,
        n_head,
        max_len,
        n_layers,
        dropout,
        device
    ):
        super().__init__()

        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.trg_sos_idx = trg_sos_idx

        self.device = device

        self.encoder = Encoder(
            enc_voc_size=enc_voc_size,
            max_len=max_len,
            d_model=d_model,
            hidden=hidden,
            n_head=n_head,
            n_layers=n_layers,
            dropout=dropout,
            device=device
        )

        self.decoder = Decoder(
            dec_voc_size=dec_voc_size,
            max_len=max_len,
            d_model=d_model,
            hidden=hidden,
            n_head=n_head,
            n_layers=n_layers,
            dropout=dropout,
            device=device
        )

    def make_src_mask(self, src):

        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)

        return src_mask

    def make_trg_mask(self, trg):

        trg_pad_mask = (trg != self.trg_pad_idx).unsqueeze(1).unsqueeze(3)

        trg_len = trg.shape[1]

        trg_sub_mask = torch.tril(torch.ones(trg_len, trg_len, dtype=torch.bool, device=self.device))

        trg_mask = trg_pad_mask & trg_sub_mask

        return trg_mask

    def forward(self, src, trg):

        src_mask = self.make_src_mask(src)

        trg_mask = self.make_trg_mask(trg)

        enc_src = self.encoder(src, src_mask)

        output = self.decoder(
            trg,
            enc_src,
            trg_mask,
            src_mask
        )

        return output

In [178]:
src=torch.randint(0,100,(2,10)).to(device)
trg=torch.randint(0,100,(2,8)).to(device)
model = Transformer(
    src_pad_idx=1,
    trg_pad_idx=1,
    trg_sos_idx=2,
    enc_voc_size=100,
    dec_voc_size=100,
    d_model=512,
    hidden=2048,
    n_head=8,
    max_len=100,
    n_layers=6,
    dropout=0.1,
    device=device
).to(device)
out=model(src,trg)
print(out.shape)

torch.Size([2, 8, 100])


In [179]:
!pip install datasets spacy torchtext

In [180]:
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 75.3 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 82.5 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [181]:
import spacy
spacy_de=spacy.load("de_core_news_sm")
spacy_en=spacy.load("en_core_web_sm")

In [182]:
def tokenize_de(text):
    return [tok.text.lower() for tok in spacy_de.tokenizer(text)]

def tokenize_en(text):
    return [tok.text.lower() for tok in spacy_en.tokenizer(text)]

In [183]:
from datasets import load_dataset
ds = load_dataset("bentrevett/multi30k")
from collections import Counter
def build_vocab(data, lang, tokenize_fn, specials, min_freq=2):
    counter = Counter()
    for example in data:
        counter.update(tokenize_fn(example[lang]))
    vocab = {tok: idx for idx, tok in enumerate(specials)}
    for token, freq in counter.items():
        if freq >= min_freq and token not in vocab:
            vocab[token] = len(vocab)
    return vocab

specials = ["<unk>", "<pad>", "<sos>", "<eos>"]

vocab_de = build_vocab(ds["train"], "de", tokenize_de, specials, min_freq=2)
vocab_en = build_vocab(ds["train"], "en", tokenize_en, specials, min_freq=2)

UNK_IDX = 0
PAD_IDX = 1
SOS_IDX = 2
EOS_IDX = 3

def text_to_indices(text, vocab, tokenize_fn):
    tokens = ["<sos>"] + tokenize_fn(text) + ["<eos>"]
    return [vocab.get(tok, UNK_IDX) for tok in tokens]

In [184]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

MAX_LEN = 128
BATCH_SIZE = 128

def text_to_tensor(text, vocab, tokenize_fn):
    indices = text_to_indices(text, vocab, tokenize_fn)
    indices = indices[:MAX_LEN]
    return torch.tensor(indices, dtype=torch.long)

def collate_fn(batch):
    src_batch, trg_batch = [], []
    for example in batch:
        src_batch.append(text_to_tensor(example["de"], vocab_de, tokenize_de))
        trg_batch.append(text_to_tensor(example["en"], vocab_en, tokenize_en))
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    trg_batch = pad_sequence(trg_batch, padding_value=PAD_IDX, batch_first=True)
    return src_batch, trg_batch

train_loader = DataLoader(ds["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(ds["validation"], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(ds["test"], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")

Train batches: 227


In [185]:
model = Transformer(
    src_pad_idx=PAD_IDX,
    trg_pad_idx=PAD_IDX,
    trg_sos_idx=SOS_IDX,
    enc_voc_size=len(vocab_de),
    dec_voc_size=len(vocab_en),
    d_model=256,
    hidden=512,
    n_head=8,
    max_len=MAX_LEN,
    n_layers=3,
    dropout=0.1,
    device=device
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

Trainable parameters: 8,987,141


In [186]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)

In [187]:
def train_epoch(model, loader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    for src, trg in loader:
        src = src.to(device)
        trg = trg.to(device)
        trg_input  = trg[:, :-1]
        trg_output = trg[:, 1:]
        optimizer.zero_grad()
        output = model(src, trg_input)
        output_flat = output.reshape(-1, output.shape[-1])
        trg_flat    = trg_output.reshape(-1)
        loss = criterion(output_flat, trg_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
        non_pad = trg_flat != PAD_IDX
        correct += output_flat.argmax(1)[non_pad].eq(trg_flat[non_pad]).sum().item()
        total   += non_pad.sum().item()
    return epoch_loss / len(loader), correct / total

In [188]:
def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for src, trg in loader:
            src = src.to(device)
            trg = trg.to(device)
            trg_input  = trg[:, :-1]
            trg_output = trg[:, 1:]
            output = model(src, trg_input)
            output_flat = output.reshape(-1, output.shape[-1])
            trg_flat = trg_output.reshape(-1)
            loss = criterion(output_flat, trg_flat)
            epoch_loss += loss.item()
            non_pad = trg_flat != PAD_IDX
            correct += output_flat.argmax(1)[non_pad].eq(trg_flat[non_pad]).sum().item()
            total   += non_pad.sum().item()
    return epoch_loss / len(loader), correct / total

In [189]:
N_EPOCHS = 20
best_val_loss = float("inf")

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_transformer.pt")
    print(f"Epoch {epoch+1:02} | Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.3f} | Val Acc: {val_acc*100:.2f}%")

Epoch 01 | Train Loss: 5.229 | Train Acc: 27.64% | Val Loss: 4.063 | Val Acc: 37.12%
Epoch 02 | Train Loss: 3.884 | Train Acc: 40.49% | Val Loss: 3.478 | Val Acc: 45.20%
Epoch 03 | Train Loss: 3.464 | Train Acc: 45.35% | Val Loss: 3.139 | Val Acc: 49.43%
Epoch 04 | Train Loss: 3.191 | Train Acc: 48.45% | Val Loss: 2.923 | Val Acc: 51.62%
Epoch 05 | Train Loss: 2.990 | Train Acc: 50.63% | Val Loss: 2.765 | Val Acc: 53.23%
Epoch 06 | Train Loss: 2.832 | Train Acc: 52.47% | Val Loss: 2.630 | Val Acc: 54.75%
Epoch 07 | Train Loss: 2.699 | Train Acc: 54.01% | Val Loss: 2.521 | Val Acc: 56.36%
Epoch 08 | Train Loss: 2.586 | Train Acc: 55.26% | Val Loss: 2.430 | Val Acc: 57.38%
Epoch 09 | Train Loss: 2.483 | Train Acc: 56.42% | Val Loss: 2.355 | Val Acc: 58.23%
Epoch 10 | Train Loss: 2.395 | Train Acc: 57.48% | Val Loss: 2.288 | Val Acc: 59.28%
Epoch 11 | Train Loss: 2.315 | Train Acc: 58.28% | Val Loss: 2.240 | Val Acc: 59.82%
Epoch 12 | Train Loss: 2.241 | Train Acc: 59.18% | Val Loss: 2.17

In [190]:
# Save the model
torch.save(model.state_dict(), "best_transformer.pt")
print("Model saved!")

# Load the model
model.load_state_dict(torch.load("best_transformer.pt", map_location=device))
model.eval()
print("Model loaded!")

# Translate function
def translate(sentence):
    model.eval()
    src = text_to_tensor(sentence, vocab_de, tokenize_de).unsqueeze(0).to(device)
    src_mask = model.make_src_mask(src)
    with torch.no_grad():
        enc_src = model.encoder(src, src_mask)
    trg_indices = [SOS_IDX]
    for _ in range(MAX_LEN):
        trg = torch.tensor(trg_indices, dtype=torch.long).unsqueeze(0).to(device)
        trg_mask = model.make_trg_mask(trg)
        with torch.no_grad():
            output = model.decoder(trg, enc_src, trg_mask, src_mask)
        pred_token = output.argmax(2)[:, -1].item()
        trg_indices.append(pred_token)
        if pred_token == EOS_IDX:
            break
    idx_to_en = {idx: tok for tok, idx in vocab_en.items()}
    tokens = [idx_to_en.get(i, "<unk>") for i in trg_indices[1:-1]]
    return " ".join(tokens)

# Test sentences
test_sentences = [
    "ein mann spielt gitarre .",
    "eine frau läuft durch den park .",
    "zwei kinder spielen im schnee .",
    "ein hund rennt über das feld .",
    "eine gruppe von menschen sitzt am strand .",
    "ein junge springt in das wasser .",
    "eine frau liest ein buch .",
    "ein mann und eine frau tanzen .",
]

print("\n" + "="*60)
print("TRANSLATION RESULTS")
print("="*60)
for sentence in test_sentences:
    translation = translate(sentence)
    print(f"DE: {sentence}")
    print(f"EN: {translation}")
    print("-"*60)

Model saved!
Model loaded!

TRANSLATION RESULTS
DE: ein mann spielt gitarre .
EN: a man is playing a guitar .
------------------------------------------------------------
DE: eine frau läuft durch den park .
EN: a woman is walking through the park .
------------------------------------------------------------
DE: zwei kinder spielen im schnee .
EN: two children playing in the snow .
------------------------------------------------------------
DE: ein hund rennt über das feld .
EN: a dog runs across the field .
------------------------------------------------------------
DE: eine gruppe von menschen sitzt am strand .
EN: a group of people sitting on the beach .
------------------------------------------------------------
DE: ein junge springt in das wasser .
EN: a boy jumps into the water .
------------------------------------------------------------
DE: eine frau liest ein buch .
EN: a woman reads a book .
------------------------------------------------------------
DE: ein mann und ei